In [7]:
import pandas as pd

df = pd.read_csv("../data/processed/model_ready_dataset.csv")
df["hour_utc"] = pd.to_datetime(df["hour_utc"], utc=True)

# NEW: drop rows where the new lag_24 feature is missing
df = df.dropna(subset=["pm25_lag_24"]).reset_index(drop=True)

df = df.sort_values(["location_name", "hour_utc"]).reset_index(drop=True)

df["row_in_station"] = df.groupby("location_name").cumcount()
df["station_size"] = df.groupby("location_name")["location_name"].transform("count")
df["split"] = (df["row_in_station"] < 0.8 * df["station_size"]).map({True: "train", False: "test"})

station = "R K Puram, Delhi - DPCC"
station_df = df[df["location_name"] == station].copy()

train_df = station_df[station_df["split"] == "train"].copy()
test_df  = station_df[station_df["split"] == "test"].copy()

In [8]:
print(train_df.shape, test_df.shape)
print("Train ends:", train_df["hour_utc"].max())
print("Test starts:", test_df["hour_utc"].min())

(1016, 29) (253, 29)
Train ends: 2026-05-22 17:00:00+00:00
Test starts: 2026-05-22 19:00:00+00:00


In [9]:
feature_cols = [
    "pm25",
    "temperature_2m", "relative_humidity_2m",
    "wind_speed_10m", "wind_direction_10m", "precipitation",
    "hour_of_day", "day_of_week", "month", "is_weekend",
    "pm25_lag_1", "pm25_lag_3", "pm25_lag_6", "pm25_lag_24",   # added
    "pm25_rolling_3", "pm25_rolling_6", "pm25_rolling_12",
]
target_col = "pm25_24h_ahead"

X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_test  = test_df[feature_cols]
y_test  = test_df[target_col]

In [10]:
print("X_train:", X_train.shape, " y_train:", y_train.shape)
print("X_test: ", X_test.shape,  " y_test: ", y_test.shape)

# Leakage check: target column MUST NOT be in features
assert target_col not in X_train.columns, "LEAKAGE: target is in features"

# Missing value check
print("NaNs in X_train:", X_train.isna().sum().sum())
print("NaNs in y_train:", y_train.isna().sum())

X_train: (1016, 17)  y_train: (1016,)
X_test:  (253, 17)  y_test:  (253,)
NaNs in X_train: 0
NaNs in y_train: 0


In [11]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_test_pred  = model.predict(X_test)

train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae  = mean_absolute_error(y_test, y_test_pred)

print(f"Train MAE: {train_mae:.2f} µg/m³")
print(f"Test MAE:  {test_mae:.2f} µg/m³")
print(f"Baseline (R K Puram): 25.34 µg/m³")

Train MAE: 4.96 µg/m³
Test MAE:  25.07 µg/m³
Baseline (R K Puram): 25.34 µg/m³


In [12]:
import joblib
import os

os.makedirs("../models", exist_ok=True)
joblib.dump(model, "../models/xgb_first_station_rkpuram.pkl")
print("Saved.")

Saved.
